# 1. Cài đặt các package cần thiết

In [30]:
# Đợi vài phút để PyPI cập nhật
!pip install --upgrade protonx
!pip install python-dotenv
!pip install requests
!pip install matplotlib
!pip install pandas
!pip install datasets
!pip install langchain-google-genai==2.1.9


# 2. Request tới Agent, lấy response để đánh giá

In [32]:
%run build_eval_dataset.py --raw-file raw_qa.txt --output-file eval_dataset.json --orchestrator-url http://localhost:7010 --sleep 5


INFO:root:Processing Q1: Stress có thể biểu hiện như thế nào về mặt cảm xúc...
INFO:root:Received response for Q1
INFO:root:Processing Q2: Stress ảnh hưởng như thế nào đến hành vi?...
INFO:root:Received response for Q2
INFO:root:Processing Q3: Stress có phải lúc nào cũng xấu không?...
INFO:root:Received response for Q3
INFO:root:Processing Q4: Khi bị stress, cơ thể thường có phản ứng ra sao?...
INFO:root:Received response for Q4
INFO:root:Processing Q5: Sinh viên thường bị stress trong những tình huống ...
INFO:root:Received response for Q5
INFO:root:Processing Q6: Làm thế nào để phòng ngừa và kiểm soát stress hiệu...
INFO:root:Received response for Q6
INFO:root:Processing Q7: Lo âu là gì?...
INFO:root:Received response for Q7
INFO:root:Processing Q8: Lo âu có thể xuất phát từ đâu?...
INFO:root:Received response for Q8
INFO:root:Processing Q9: Người bị lo âu thường có những biểu hiện tâm lý nà...
INFO:root:Received response for Q9
INFO:root:Processing Q10: Khi bị lo âu, cơ thể thường p

[OK] Saved evaluation dataset: eval_dataset.json (samples=10)


In [4]:
from protonx import ProtonX

In [5]:
import protonx
protonx.__version__

'0.1.6'

#### Bạn hãy lấy Access Token của mình [ở đây](https://platform.protonx.io/).
#### Set API Key: PROTONX_API_KEY=api_key

In [17]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [18]:
client = ProtonX()

## Kiểm tra tính chính xác của chatbot

Điểm liên quan tới chất lượng trả lời
- Precision
- Recall
- F1



In [34]:
metrics = ["precision", "recall", "f1"]

In [33]:
import json
from pathlib import Path

dataset_path = Path("eval_dataset.json")
data = json.loads(dataset_path.read_text(encoding="utf-8"))
data[:3]


[{'question': 'Stress có thể biểu hiện như thế nào về mặt cảm xúc?',
  'ground_truth': 'Stress thường biểu hiện rõ rệt qua những thay đổi về mặt cảm xúc. Người đang chịu căng thẳng có thể cảm thấy khó chịu, bực bội hoặc dễ cáu gắt mà không rõ lý do. Họ thường xuyên lo lắng, bất an, luôn trong trạng thái căng thẳng. Nhiều người rơi vào cảm giác buồn bã, chán nản, mất hứng thú với những điều từng yêu thích. Ở mức độ cao hơn, stress còn khiến họ cảm thấy thờ ơ với mọi thứ xung quanh, mất niềm tin vào bản thân và cảm giác như đánh mất giá trị của chính mình.',
  'answer': 'Chào bạn,\n\nBạn đang thắc mắc về việc stress có thể biểu hiện như thế nào về mặt cảm xúc đúng không? Mình hiểu rằng đôi khi những cảm xúc tiêu cực có thể khiến chúng ta cảm thấy khó khăn và bối rối. \n\nVề mặt cảm xúc, stress có thể hiện ra rất nhiều cách khác nhau, có thể khiến bạn cảm thấy:\n\n*   **Khó chịu:** Cảm giác bực bội, khó chịu, dễ cáu gắt hơn bình thường.\n*   **Lo lắng, căng thẳng:** Luôn cảm thấy bồn chồn

Câu hỏi mà bạn đưa vào Chatbot của bạn

In [35]:
questions = [sample["question"] for sample in data]  # nếu dùng protonx_dataset
# hoặc nếu dùng eval_dataset.json (chưa map):
# questions = [sample["question"] for sample in data]

questions[:3]


['Stress có thể biểu hiện như thế nào về mặt cảm xúc?',
 'Stress ảnh hưởng như thế nào đến hành vi?',
 'Stress có phải lúc nào cũng xấu không?']

Câu trả lời bạn kỳ vọng chatbot sẽ trả lời

In [36]:
ground_truths = [sample["ground_truth"] for sample in data]

len(ground_truths), ground_truths[:3]

(10,
 ['Stress thường biểu hiện rõ rệt qua những thay đổi về mặt cảm xúc. Người đang chịu căng thẳng có thể cảm thấy khó chịu, bực bội hoặc dễ cáu gắt mà không rõ lý do. Họ thường xuyên lo lắng, bất an, luôn trong trạng thái căng thẳng. Nhiều người rơi vào cảm giác buồn bã, chán nản, mất hứng thú với những điều từng yêu thích. Ở mức độ cao hơn, stress còn khiến họ cảm thấy thờ ơ với mọi thứ xung quanh, mất niềm tin vào bản thân và cảm giác như đánh mất giá trị của chính mình.',
  'Stress có thể tác động mạnh mẽ đến hành vi của con người, khiến họ thay đổi cách ứng xử trong cuộc sống hằng ngày. Một người đang chịu căng thẳng thường dễ nổi cáu, bực bội hoặc trở nên nóng nảy hơn bình thường. Họ có xu hướng tìm đến các chất kích thích như rượu, thuốc lá để giải tỏa tạm thời. Các thói quen sinh hoạt cũng dễ bị xáo trộn — ăn uống thất thường, mất ngủ hoặc ngủ quá nhiều. Ngoài ra, stress còn khiến con người mất tập trung, hay quên, đưa ra những quyết định thiếu lý trí, hành động vội vàng, hấp

Ngữ cảnh chatbot dùng để trả lời

In [37]:
contexts = [sample["answer"] for sample in data]

len(contexts), contexts[:3]

(10,
 ['Chào bạn,\n\nBạn đang thắc mắc về việc stress có thể biểu hiện như thế nào về mặt cảm xúc đúng không? Mình hiểu rằng đôi khi những cảm xúc tiêu cực có thể khiến chúng ta cảm thấy khó khăn và bối rối. \n\nVề mặt cảm xúc, stress có thể hiện ra rất nhiều cách khác nhau, có thể khiến bạn cảm thấy:\n\n*   **Khó chịu:** Cảm giác bực bội, khó chịu, dễ cáu gắt hơn bình thường.\n*   **Lo lắng, căng thẳng:** Luôn cảm thấy bồn chồn, lo lắng về những việc sắp tới hoặc những vấn đề đang gặp phải.\n*   **Buồn bã, chán nản:** Mất hứng thú với những việc mình từng yêu thích, cảm thấy mọi thứ trở nên vô vị.\n*   **Mất giá trị bản thân:** Cảm thấy mình không đủ tốt, không có giá trị, dễ tự ti.\n\nNgoài ra, stress còn có thể khiến bạn dễ nổi nóng, trở nên nóng tính hơn, hoặc thậm chí là cảm thấy mất mát, trống rỗng. \n\nViệc nhận biết những biểu hiện cảm xúc này là bước đầu tiên rất quan trọng để đối diện với stress. Hãy thử dành thời gian lắng nghe cảm xúc của mình, chấp nhận rằng việc cảm thấy 

Câu trả lời thực tế của Chatbot

In [38]:
response_llms = [sample["answer"] for sample in data]

len(response_llms), response_llms[:3]

(10,
 ['Chào bạn,\n\nBạn đang thắc mắc về việc stress có thể biểu hiện như thế nào về mặt cảm xúc đúng không? Mình hiểu rằng đôi khi những cảm xúc tiêu cực có thể khiến chúng ta cảm thấy khó khăn và bối rối. \n\nVề mặt cảm xúc, stress có thể hiện ra rất nhiều cách khác nhau, có thể khiến bạn cảm thấy:\n\n*   **Khó chịu:** Cảm giác bực bội, khó chịu, dễ cáu gắt hơn bình thường.\n*   **Lo lắng, căng thẳng:** Luôn cảm thấy bồn chồn, lo lắng về những việc sắp tới hoặc những vấn đề đang gặp phải.\n*   **Buồn bã, chán nản:** Mất hứng thú với những việc mình từng yêu thích, cảm thấy mọi thứ trở nên vô vị.\n*   **Mất giá trị bản thân:** Cảm thấy mình không đủ tốt, không có giá trị, dễ tự ti.\n\nNgoài ra, stress còn có thể khiến bạn dễ nổi nóng, trở nên nóng tính hơn, hoặc thậm chí là cảm thấy mất mát, trống rỗng. \n\nViệc nhận biết những biểu hiện cảm xúc này là bước đầu tiên rất quan trọng để đối diện với stress. Hãy thử dành thời gian lắng nghe cảm xúc của mình, chấp nhận rằng việc cảm thấy 

In [39]:
evaluator = client.evals()
evaluator.eval_llm_answer(
    metrics=metrics,
    questions=questions,
    response_llms=response_llms,
    contexts=contexts,
    ground_truths=ground_truths)

APIError: APIError 429: You have exceeded the daily request limit (10). Please try again later.